In [ ]:
!pip install langchain langchain_community langchain_openai pypdf faiss-cpu gradio


In [ ]:
from google.colab import userdata


In [ ]:
OPENAI_API_KEY = userdata.get('openai-api')
TAVILY_API_KEY = userdata.get('travily-api')

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import os
import gradio as gr
from gradio import ChatMessage
from dotenv import load_dotenv

In [3]:
load_dotenv() # .env 파일에서 환경변수 로드
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [6]:

llm = ChatOpenAI(temperature=1.0, model='gpt-4o-mini')  

# LLM 응답 처리
def response(message, history, additional_input_info): # 사용자 입력 메시지, 이전 대화기록, 시스템 메시지
    history_langchain_format = []
    if additional_input_info:
        history_langchain_format.append(SystemMessage(content=additional_input_info))
    
    for msg in history:
        if isinstance(msg, ChatMessage):
            if msg.role == "user":
                history_langchain_format.append(HumanMessage(content=msg.content))
            else:
                history_langchain_format.append(AIMessage(content=msg.content))
        elif isinstance(msg, (list, tuple)) and len(msg) == 2:
            history_langchain_format.append(HumanMessage(content=msg[0]))
            history_langchain_format.append(AIMessage(content=msg[1]))
    
    history_langchain_format.append(HumanMessage(content=message))
    gpt_response = llm.invoke(history_langchain_format)
    return gpt_response.content

# 인터페이스 생성
gr.ChatInterface(
    fn=response,   # LLM 응답처리 콜백함수 설정
    textbox=gr.Textbox(placeholder="Talk", container=False, scale=7),
    chatbot=gr.Chatbot(height=500),
    title="ChatBot",
    description="I'm a chatbot that can chat with you. I'm lovely chatbot.",
    examples=[["Hi"], ["I'm good"], ["What's your name?"]],
    additional_inputs=[
        gr.Textbox("", label="Input System Prompt", placeholder="I'm chatbot.") # placeholder="I'm chatbot." or "I'm a chatbot that can chat with you." or "I'm japanese chatbot."
    ]
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
